In [ ]:
# --- imports + fixed fiducial cosmology (lhid=2000, 0-indexed, from latin_hypercube_params.txt) ---
import numpy as np
import pandas as pd
import h5py
import os
from os.path import join, exists
from tqdm import tqdm
from colossus.cosmology import cosmology
from colossus.halo import mass_so

# 3 fiducial 3Gpc/h sims, all at the same cosmology (line 2000, 0-indexed)
sim_id = 2000
IN_TMPL = f'/work/hdd/bdne/maho3/quijote/Quijote_3gpc_HR/{sim_id}/out_1_pid.list'
OUT_TMPL = f'/work/hdd/bdne/maho3/cmass-ili/quijote3gpch/nbody/L3000-N384/{sim_id}'

# lhid=2000 fiducial cosmology: Om0, Ob0, h0, ns, sigma8
FIDUCIAL_COSMO = (0.3175, 0.04900, 0.6711, 0.9624, 0.834)

In [6]:
# --- loader: M200c mass, R200c/Rs concentration, parents only, mass cut ---
def load_rockstar_list(path):
    a = 0.666667                                   # snapshot 3
    z = 1.0 / a - 1.0

    with open(path) as f:                          # read column-name header
        header = f.readline().lstrip('#').strip().split()
    df = pd.read_csv(path, sep=r'\s+', comment='#', header=None,
                     names=header, engine='c')

    # parents only (drop subhalos)
    df = df[df['PID'] == -1]

    pos = df[['X', 'Y', 'Z']].to_numpy(np.float32)    # Mpc/h comoving
    vel = df[['VX', 'VY', 'VZ']].to_numpy(np.float32)  # km/s physical
    M200c = df['M200c'].to_numpy(np.float64)          # Msun/h
    Rs = df['Rs'].to_numpy(np.float64)                # kpc/h comoving

    Om0, Ob0, h0, ns_cosmo, sigma8 = FIDUCIAL_COSMO
    cosmology.setCosmology('myCosmo', **{
        'flat': True, 'H0': h0 * 100, 'Om0': Om0, 'Ob0': Ob0,
        'sigma8': sigma8, 'ns': ns_cosmo})

    R200c = mass_so.M_to_R(M200c, z, '200c') * (1.0 + z)  # -> comoving kpc/h
    conc = (R200c / Rs).astype(np.float32)
    with np.errstate(divide='ignore'):
        mass = np.log10(M200c.astype(np.float32))         # log10(Msun/h)

    m = mass >= np.log10(5e12)                            # CHARM mass cut
    return dict(mass=mass[m], pos=pos[m], vel=vel[m], conc=conc[m], a=a)

In [7]:
def save_snapshot(outdir, a, hpos, hvel, hmass, **meta):
    # from rho_to_halo
    with h5py.File(join(outdir, 'halos.h5'), 'a') as f:
        group = f.create_group(f'{a:.6f}')
        group.create_dataset('pos', data=hpos)  # comoving positions [Mpc/h]
        group.create_dataset('vel', data=hvel)  # physical velocities [km/s]
        group.create_dataset('mass', data=hmass)  # halo masses [Msun/h]

        # save other halo metadata (e.g. concentration)
        for key, val in meta.items():
            group.create_dataset(key, data=val)

In [9]:
# --- process 3 fiducial 3Gpc/h sims sequentially ---
N_SIMS = 1

counts = {'done': 0, 'skip': 0, 'miss': 0, 'err': 0}
for sim_id in tqdm(range(N_SIMS), desc='sims'):
    path = IN_TMPL.format(sim_id=sim_id)
    outdir = OUT_TMPL.format(sim_id=sim_id)

    if exists(join(outdir, 'halos.h5')):
        counts['skip'] += 1
        continue
    if not exists(path):
        counts['miss'] += 1
        tqdm.write(f'[{sim_id}] MISS {path}')
        continue

    try:
        halos = load_rockstar_list(path)
        os.makedirs(outdir, exist_ok=True)
        save_snapshot(outdir, a=halos['a'], hpos=halos['pos'],
                      hvel=halos['vel'], hmass=halos['mass'],
                      concentration=halos['conc'])
        counts['done'] += 1
    except Exception as e:
        counts['err'] += 1
        tqdm.write(f'[{sim_id}] ERR {type(e).__name__}: {e}')

print(f"done: ok={counts['done']} skip={counts['skip']} "
      f"miss={counts['miss']} err={counts['err']}")

sims: 100%|██████████| 1/1 [09:45<00:00, 585.60s/it]

done: ok=1 skip=0 miss=0 err=0
